# Chat Response Comparator

Results can be saved to an MLflow server. This notebook compares model chat responses with a reference `ground_truth` file and produces one stable run.

## Version 10.2 - English JSON schema

What's new:
1. Uses one `ground_truth*.json`, the default Polish prompt matching `promptEN*.md`, and the first sorted `input*.json` file.
2. Merges all `chat_response*.json` files into one combined response set.
3. Uses an English-only JSON schema for requirements, expert labels, and LLM responses.
4. Computes metrics, writes a Markdown report, and can log metrics/artifacts to MLflow.

## How to run

1. **Clone the notebook before editing.**
   - Click **File -> Save a copy in Drive**.
   - Work on your copy; the original may be read-only.

2. **Add files to the working directory:**
   - `ground_truth*.json` - reference answers file with `id` and `expert_classification`.
   - `chat_response*.json` - chat answer files with `id`, `classification`, `confidence`, `source`, and optional `justification`; multiple files are merged into one run.
- `prompt*.md` - prompt or instruction file used by default `prompt.md` (English variant available as `prompt_eng.md`).
   - `input*.json` - requirements file with `id`, `name`, and `description`; the first sorted match is used.

3. **Expected JSON fields:**
   - Input requirement: `{ "id": "REQ-001", "name": "Requirement name", "description": "Requirement description" }`
   - Ground truth: `{ "id": "REQ-001", "expert_classification": "STANDARD" }`
   - LLM response: `{ "id": "REQ-001", "classification": "STANDARD", "confidence": 5, "source": ["..."], "justification": "..." }`

4. **Set MLflow parameters:**
   - `PLATFORM = 'Claude'` - platform name, e.g. Claude, GPT, Gemini.
   - `MODEL = 'Haiku 4.5'` - model name.
   - `PROMPT_VERSION = '9'` - prompt version, important for stability studies.
   - `PROJECT_SPACE = 'ERP XL Requirements Analysis'` - project name.
   - `DOCUMENTATION = 'ComarchDocumentation.md'` - documentation filename.
   - `AUTHOR = 'Your Name'`.

5. **Click "Run All".**

6. **Results will be saved in** `results_DATE_TIME/`:
   - `report.md` - human-readable report.
   - `combined_chat_responses.json` - merged chat responses used for the run.


## Install Dependencies


In [ ]:
!pip install --upgrade pip -q
!pip install pandas seaborn matplotlib scikit-learn tabulate mlflow -q


## Import Libraries


In [2]:
# === Import Libraries ===
import os
import glob
import json
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from datetime import datetime
import tempfile
import shutil
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score, accuracy_score
)


## Configuration


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                    FILE CONFIGURATION                           ║
# ╚══════════════════════════════════════════════════════════════════╝

GROUND_TRUTH = 'ground_truth*.json'       # REQUIRED - reference answers file
CHAT_RESPONSES = 'chat_response*.json'    # REQUIRED - response files merged into one run
PROMPT_FILE = 'prompt*.md'                # REQUIRED - default prompt file
INPUT_FILE = 'input*.json'                # REQUIRED - first sorted match is used

GROUND_TRUTH_STANDARD = 'ground_truth.json'
PROMPT_FILE_STANDARD = 'prompt.md'
INPUT_FILE_STANDARD = 'input.json'
COMBINED_CHAT_RESPONSES_FILE_STANDARD = 'combined_chat_responses.json'
REPORT_FILE_STANDARD = 'report.md'

# English JSON field names
ID_FIELD = 'id'
NAME_FIELD = 'name'
DESCRIPTION_FIELD = 'description'
EXPECTED_CLASSIFICATION_FIELD = 'expert_classification'
PREDICTED_CLASSIFICATION_FIELD = 'classification'
CLASSIFICATION_FIELD = 'classification'
CONFIDENCE_FIELD = 'confidence'
SOURCE_FIELD = 'source'
JUSTIFICATION_FIELD = 'justification'

# Allowed classification categories
VALID_CATEGORIES = ['STANDARD', 'CUSTOMIZATION', 'NO_ANSWER']
NO_ANSWER = 'NO_ANSWER'


## Helper Functions


In [4]:
def normalize_value(value):
    """Return a stable string representation for comparisons and metrics."""
    if value is None:
        return ''
    return str(value).strip()


def normalize_category(value):
    value = normalize_value(value)
    return value.upper() if value else ''


def safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def finite_metric(value, default=0.0):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return default
    return default if pd.isna(value) else value


def safe_matthews_corrcoef(y_true, y_pred):
    if not y_true or len(set(y_true + y_pred)) < 2:
        return 0.0
    return finite_metric(matthews_corrcoef(y_true, y_pred))


def safe_cohen_kappa_score(y_true, y_pred):
    if not y_true or len(set(y_true + y_pred)) < 2:
        return 0.0
    return finite_metric(cohen_kappa_score(y_true, y_pred))


def build_duplicate_map(items, id_field=ID_FIELD):
    counts = Counter(normalize_value(item.get(id_field)) for item in items)
    return sorted(item_id for item_id, count in counts.items() if item_id and count > 1)


def build_requirement_index(input_records):
    """Index input requirements by English `id` for report enrichment."""
    index = {}
    invalid_records = []
    for item in input_records:
        item_id = normalize_value(item.get(ID_FIELD))
        if not item_id:
            invalid_records.append(item)
            continue
        index[item_id] = item
    return index, invalid_records


def requirement_name(item_id, ground_truth_item, requirement_index):
    input_item = requirement_index.get(item_id, {}) if requirement_index else {}
    return ground_truth_item.get(NAME_FIELD) or input_item.get(NAME_FIELD, '')


def compare_results(chat_responses, ground_truth, requirement_index=None):
    """
    Compare LLM responses with expert classifications using English JSON fields.

    Reports records missing on either side and keeps comparison rows robust to
    missing IDs, None values, casing differences, and extra spaces.
    """
    requirement_index = requirement_index or {}
    gt_index = {}
    invalid_ground_truth = []
    for item in ground_truth:
        item_id = normalize_value(item.get(ID_FIELD))
        if not item_id:
            invalid_ground_truth.append(item)
            continue
        gt_index[item_id] = item

    chat_index = {}
    invalid_chat = []
    results = []
    total_matches = 0
    total_comparisons = 0

    for chat in chat_responses:
        item_id = normalize_value(chat.get(ID_FIELD))
        if not item_id:
            invalid_chat.append(chat)
            results.append({
                ID_FIELD: '',
                'status': 'INVALID_CHAT_ID',
                'message': 'Missing id in chat response',
            })
            continue

        chat_index[item_id] = chat
        gt = gt_index.get(item_id)

        if gt is None:
            results.append({ID_FIELD: item_id, 'status': 'NOT_FOUND_IN_GT'})
            continue

        comparison = {
            ID_FIELD: item_id,
            NAME_FIELD: requirement_name(item_id, gt, requirement_index),
            CONFIDENCE_FIELD: chat.get(CONFIDENCE_FIELD, ''),
            SOURCE_FIELD: chat.get(SOURCE_FIELD, []),
            'fields': {},
        }

        chat_val = normalize_category(chat.get(PREDICTED_CLASSIFICATION_FIELD))
        gt_val = normalize_category(gt.get(EXPECTED_CLASSIFICATION_FIELD))

        if gt_val:
            match = bool(chat_val) and chat_val == gt_val
            comparison['fields'][CLASSIFICATION_FIELD] = {
                'chat': chat_val,
                'expected': gt_val,
                'match': match,
            }
            total_comparisons += 1
            if match:
                total_matches += 1

        comparison['all_match'] = total_comparisons > 0 and all(
            data['match'] for data in comparison['fields'].values()
        )
        if comparison['fields']:
            results.append(comparison)

    for item_id, gt in gt_index.items():
        if item_id not in chat_index:
            results.append({
                ID_FIELD: item_id,
                NAME_FIELD: requirement_name(item_id, gt, requirement_index),
                'status': 'NOT_IN_CHAT',
            })

    return {
        'results': results,
        'summary': {
            'total_matches': total_matches,
            'total_comparisons': total_comparisons,
            'num_chat_responses': len(chat_responses),
            'num_ground_truth': len(ground_truth),
            'num_invalid_chat_ids': len(invalid_chat),
            'num_invalid_ground_truth_ids': len(invalid_ground_truth),
            'duplicate_chat_ids': build_duplicate_map(chat_responses),
            'duplicate_ground_truth_ids': build_duplicate_map(ground_truth),
        }
    }


In [5]:
def visualize_results(comparison_result):
    """Create a colored HTML table with comparison results."""
    rows = []
    for item in comparison_result['results']:
        status = item.get('status')
        if status == 'NOT_IN_CHAT':
            rows.append({
                ID_FIELD: item[ID_FIELD],
                NAME_FIELD: item.get(NAME_FIELD, ''),
                CONFIDENCE_FIELD: '',
                CLASSIFICATION_FIELD: '⚠️ Missing in chat responses'
            })
            continue
        if status == 'NOT_FOUND_IN_GT':
            rows.append({
                ID_FIELD: item[ID_FIELD],
                NAME_FIELD: '',
                CONFIDENCE_FIELD: '',
                CLASSIFICATION_FIELD: '⚠️ Missing in ground truth'
            })
            continue
        if status == 'INVALID_CHAT_ID':
            rows.append({
                ID_FIELD: '',
                NAME_FIELD: '',
                CONFIDENCE_FIELD: '',
                CLASSIFICATION_FIELD: '⚠️ Missing id in response'
            })
            continue
        if 'fields' not in item:
            continue

        row = {ID_FIELD: item[ID_FIELD]}
        row[NAME_FIELD] = item.get(NAME_FIELD, '')
        row[CONFIDENCE_FIELD] = item.get(CONFIDENCE_FIELD, '')
        for field, data in item['fields'].items():
            if data['match']:
                row[field] = f"✅ {data['chat']}"
            elif normalize_category(data['chat']) == NO_ANSWER:
                row[field] = f"❓ Response: {data['chat']} | Expected: {data['expected']}"
            else:
                row[field] = f"❌ Response: {data['chat']} | Expected: {data['expected']}"
        rows.append(row)

    df = pd.DataFrame(rows)

    def color_cell(val):
        text = str(val)
        if '✅' in text:
            return 'background-color: #d4edda; color: #155724'
        if '❌' in text:
            return 'background-color: #f8d7da; color: #721c24'
        if '❓' in text:
            return 'background-color: #fff3cd; color: #856404'
        if '⚠️' in text:
            return 'background-color: #ffe0b2; color: #e65100'
        return ''

    return df.style.applymap(color_cell)


In [6]:
def calculate_metrics(comparison_result, field=CLASSIFICATION_FIELD):
    """Calculate classification metrics and confidence-threshold metrics."""

    all_pairs = []
    pairs_with_confidence = []
    labels = [label.lower() for label in VALID_CATEGORIES]

    for item in comparison_result['results']:
        if 'fields' not in item or field not in item['fields']:
            continue
        data = item['fields'][field]
        chat_value = normalize_category(data.get('chat')).lower()
        expected_value = normalize_category(data.get('expected')).lower()
        if chat_value and expected_value:
            pair = (chat_value, expected_value)
            all_pairs.append(pair)
            confidence = safe_int(item.get(CONFIDENCE_FIELD), default=0)
            pairs_with_confidence.append((pair, confidence))

    y_pred = [p[0] for p in all_pairs]
    y_true = [p[1] for p in all_pairs]
    observed_labels = sorted(set(y_true + y_pred))
    metric_labels = list(labels)
    metric_labels.extend(label for label in observed_labels if label not in metric_labels)

    if not all_pairs:
        empty_cm = pd.DataFrame(0, index=labels, columns=labels)
        empty_cm.index.name = 'Expected ↓ \\ Prediction →'
        return {
            'per_category': {
                label: {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0}
                for label in labels
            },
            'overall_macro': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0},
            'overall_weighted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0},
            'global': {'accuracy': 0.0, 'mcc': 0.0, 'kappa': 0.0},
            'confusion_matrix': empty_cm,
            'num_samples': 0,
            'per_confidence': {min_confidence: None for min_confidence in range(1, 6)},
        }

    report = classification_report(
        y_true, y_pred, labels=metric_labels, output_dict=True, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=metric_labels)
    df_cm = pd.DataFrame(cm, index=metric_labels, columns=metric_labels)
    df_cm.index.name = 'Expected ↓ \\ Prediction →'

    metrics_per_confidence = {}
    for min_confidence in range(1, 6):
        filtered = [(p, t) for (p, t), confidence in pairs_with_confidence if confidence >= min_confidence]
        if not filtered:
            metrics_per_confidence[min_confidence] = None
            continue
        yp = [p for p, t in filtered]
        yt = [t for p, t in filtered]
        rep = classification_report(
            yt, yp, labels=metric_labels, output_dict=True, zero_division=0
        )
        metrics_per_confidence[min_confidence] = {
            'accuracy': accuracy_score(yt, yp),
            'mcc': safe_matthews_corrcoef(yt, yp),
            'kappa': safe_cohen_kappa_score(yt, yp),
            'macro_precision': rep['macro avg']['precision'],
            'macro_recall': rep['macro avg']['recall'],
            'macro_f1': rep['macro avg']['f1-score'],
            'weighted_precision': rep['weighted avg']['precision'],
            'weighted_recall': rep['weighted avg']['recall'],
            'weighted_f1': rep['weighted avg']['f1-score'],
            'num_samples': len(filtered),
        }

    return {
        'per_category': {cat: report[cat] for cat in metric_labels},
        'overall_macro': report['macro avg'],
        'overall_weighted': report['weighted avg'],
        'global': {
            'accuracy': accuracy_score(y_true, y_pred),
            'mcc': safe_matthews_corrcoef(y_true, y_pred),
            'kappa': safe_cohen_kappa_score(y_true, y_pred),
        },
        'confusion_matrix': df_cm,
        'num_samples': len(all_pairs),
        'per_confidence': metrics_per_confidence,
    }


In [7]:
def markdown_cell(value):
    """Escape values so generated Markdown tables stay valid."""
    text = '' if value is None else str(value)
    return text.replace('\\', '\\\\').replace('|', '\\|').replace('\r', ' ').replace('\n', '<br>')


def format_percent(value):
    return f"{float(value):.2%}"


def format_float(value):
    return f"{float(value):.4f}"


def generate_report(comparison_result, metrics, filename='report.md', input_metadata=None):
    """Save results and metrics to a Markdown file."""
    input_metadata = input_metadata or {}
    summary = comparison_result.get('summary', {})

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("# Chat Response Comparator Report\n\n")

        f.write("## Input files\n\n")
        f.write(f"- **Ground truth:** {markdown_cell(input_metadata.get('ground_truth_file', ''))}\n")
        f.write(f"- **Prompt:** {markdown_cell(input_metadata.get('prompt_file', ''))}\n")
        f.write(f"- **Input:** {markdown_cell(input_metadata.get('input_file', ''))}\n")
        answer_files = input_metadata.get('answer_files', [])
        f.write(f"- **Chat response files:** {markdown_cell(', '.join(answer_files))}\n")
        f.write(f"- **Combined responses:** {markdown_cell(input_metadata.get('combined_file', ''))}\n\n")

        f.write("## Data quality\n\n")
        f.write(f"- **Compared samples:** {metrics['num_samples']}\n")
        f.write(f"- **Chat responses:** {summary.get('num_chat_responses', 0)}\n")
        f.write(f"- **Ground truth rows:** {summary.get('num_ground_truth', 0)}\n")
        f.write(f"- **Invalid chat IDs:** {summary.get('num_invalid_chat_ids', 0)}\n")
        f.write(f"- **Invalid ground truth IDs:** {summary.get('num_invalid_ground_truth_ids', 0)}\n")
        f.write(f"- **Duplicate chat IDs:** {markdown_cell(', '.join(summary.get('duplicate_chat_ids', [])) or 'none')}\n")
        f.write(f"- **Duplicate ground truth IDs:** {markdown_cell(', '.join(summary.get('duplicate_ground_truth_ids', [])) or 'none')}\n\n")

        f.write("## Metrics\n\n")
        f.write("### Macro Average\n")
        f.write(f"- **Precision:** {format_percent(metrics['overall_macro']['precision'])}\n")
        f.write(f"- **Recall:** {format_percent(metrics['overall_macro']['recall'])}\n")
        f.write(f"- **F1:** {format_percent(metrics['overall_macro']['f1-score'])}\n\n")

        f.write("### Weighted Average\n")
        f.write(f"- **Precision:** {format_percent(metrics['overall_weighted']['precision'])}\n")
        f.write(f"- **Recall:** {format_percent(metrics['overall_weighted']['recall'])}\n")
        f.write(f"- **F1:** {format_percent(metrics['overall_weighted']['f1-score'])}\n\n")

        f.write("### Global\n")
        f.write(f"- **Accuracy:** {format_percent(metrics['global']['accuracy'])}\n")
        f.write(f"- **MCC:** {format_float(metrics['global']['mcc'])}\n")
        f.write(f"- **Cohen's Kappa:** {format_float(metrics['global']['kappa'])}\n")

        f.write("\n## Confusion Matrix\n\n")
        cm = metrics['confusion_matrix'].fillna(0).astype(int)
        cm.index.name = 'Expert ↓ \\ Model →'
        f.write(cm.to_markdown())
        f.write("\n\n")

        f.write("## Metrics per category\n\n")
        f.write("| Category | Precision | Recall | F1 | Support |\n")
        f.write("|----------|-----------|--------|----|---------|\n")
        for cat, m in metrics['per_category'].items():
            f.write(
                f"| {markdown_cell(cat)} | {format_percent(m['precision'])} | "
                f"{format_percent(m['recall'])} | {format_percent(m['f1-score'])} | {m['support']} |\n"
            )

        f.write("\n## Metrics per confidence\n\n")
        f.write("| Confidence >= | Accuracy | MCC | Kappa | Prec (macro) | Recall (macro) | F1 (macro) | Prec (wt) | Recall (wt) | F1 (wt) | N |\n")
        f.write("|---------------|----------|-----|-------|--------------|----------------|------------|-----------|-------------|---------|---|\n")
        for threshold, m in metrics['per_confidence'].items():
            if m:
                f.write(
                    f"| {threshold} | {format_percent(m['accuracy'])} | {format_float(m['mcc'])} | "
                    f"{format_float(m['kappa'])} | {format_percent(m['macro_precision'])} | "
                    f"{format_percent(m['macro_recall'])} | {format_percent(m['macro_f1'])} | "
                    f"{format_percent(m['weighted_precision'])} | {format_percent(m['weighted_recall'])} | "
                    f"{format_percent(m['weighted_f1'])} | {m['num_samples']} |\n"
                )
            else:
                f.write(f"| {threshold} | n/a | n/a | n/a | n/a | n/a | n/a | n/a | n/a | n/a | 0 |\n")

        counts = defaultdict(lambda: {"YES": 0, "NO": 0, "NO_ANSWER": 0})
        status_counts = Counter()

        for item in comparison_result['results']:
            status = item.get('status')
            if status:
                status_counts[status] += 1
            if 'fields' not in item:
                continue

            confidence = item.get(CONFIDENCE_FIELD, '')
            for field, data in item['fields'].items():
                if data['match']:
                    status_label = "YES"
                elif normalize_category(data['chat']) == NO_ANSWER:
                    status_label = "NO_ANSWER"
                else:
                    status_label = "NO"
                counts[confidence][status_label] += 1

        f.write("\n## Missing and invalid rows\n\n")
        f.write("| Status | Count |\n")
        f.write("|--------|-------|\n")
        for status, count in sorted(status_counts.items()):
            f.write(f"| {markdown_cell(status)} | {count} |\n")

        f.write("\n## Match count by confidence\n\n")
        f.write("| Confidence | YES | NO | NO_ANSWER | Total |\n")
        f.write("|------------|-----|----|-----------|-------|\n")
        for confidence, vals in sorted(counts.items(), key=lambda x: str(x[0])):
            total = vals["YES"] + vals["NO"] + vals["NO_ANSWER"]
            f.write(f"| {markdown_cell(confidence)} | {vals['YES']} | {vals['NO']} | {vals['NO_ANSWER']} | {total} |\n")

        f.write("\n## Detailed results\n\n")
        f.write("| ID | Name | Chat | Expected | Confidence | Match | Status |\n")
        f.write("|----|------|------|----------|------------|-------|--------|\n")

        for item in comparison_result['results']:
            if 'fields' not in item:
                f.write(
                    f"| {markdown_cell(item.get(ID_FIELD, ''))} | {markdown_cell(item.get(NAME_FIELD, ''))} | "
                    f" | | | | {markdown_cell(item.get('status', ''))} |\n"
                )
                continue

            for field, data in item['fields'].items():
                if data['match']:
                    status = "YES"
                elif normalize_category(data['chat']) == NO_ANSWER:
                    status = "NO_ANSWER"
                else:
                    status = "NO"

                f.write(
                    f"| {markdown_cell(item[ID_FIELD])} | {markdown_cell(item.get(NAME_FIELD, ''))} | "
                    f"{markdown_cell(data['chat'])} | {markdown_cell(data['expected'])} | "
                    f"{markdown_cell(item.get(CONFIDENCE_FIELD, ''))} | {status} | |\n"
                )

    print(f"Saved to {filename}")


In [8]:
# English-only JSON schema: no key renaming is applied.


In [9]:
def plot_confusion_matrix(metrics):
    """Display the confusion matrix as a heatmap."""
    cm = metrics['confusion_matrix']

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.gca().xaxis.set_label_position('top')
    plt.gca().tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
    plt.xlabel('Prediction')
    plt.ylabel('Expected')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()


# Load and Save Data


In [10]:
def check_files(mask, missing_message):
    """Return sorted files matching a glob mask or stop the notebook with a clear error."""
    files = sorted(glob.glob(mask))
    if not files:
        print(missing_message)
        print("   Make sure the files are in the working directory.")
        sys.exit(1)
    print(f"✅ Found {len(files)} file(s): {files}")
    return files


def check_file(mask, missing_message):
    """Return the first sorted file matching a glob mask."""
    files = check_files(mask, missing_message)
    selected = files[0]
    if len(files) > 1:
        print(f"ℹ️ Using first sorted match for {mask}: {selected}")
    return selected


def load_json_records(filepath):
    """Load a JSON file as a list of records."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data if isinstance(data, list) else [data]


# === FILE VALIDATION ===
truth_file = check_file(
    GROUND_TRUTH,
    f"❌ Error: reference file not found: {GROUND_TRUTH}"
)
ground_truth = load_json_records(truth_file)

prompt_file = check_file(
    PROMPT_FILE,
    f"❌ Error: prompt file not found: {PROMPT_FILE}"
)

input_files = check_files(
    INPUT_FILE,
    f"❌ Error: input files not found: {INPUT_FILE}"
)
input_file = input_files[0]
if len(input_files) > 1:
    print(f"ℹ️ Using first sorted input file: {input_file}")
input_records = load_json_records(input_file)
requirement_index, invalid_input_records = build_requirement_index(input_records)
if invalid_input_records:
    print(f"⚠️ Input records without id: {len(invalid_input_records)}")

chat_files = check_files(
    CHAT_RESPONSES,
    f"❌ Error: chat response files not found: {CHAT_RESPONSES}"
)

chat_responses = []
for filepath in chat_files:
    chat_responses.extend(load_json_records(filepath))

# === RESULTS FOLDER ===
today = datetime.now().strftime('%Y-%m-%d_%H-%M')
results_folder = f'results_{today}'
os.makedirs(results_folder, exist_ok=True)

combined_chat_responses_path = f'{results_folder}/{COMBINED_CHAT_RESPONSES_FILE_STANDARD}'
with open(combined_chat_responses_path, 'w', encoding='utf-8') as f:
    json.dump(chat_responses, f, ensure_ascii=False, indent=2)

print(f"✅ Using input file: {input_file}")
print(f"✅ Saved combined responses to: {combined_chat_responses_path}")


❌ Error: reference file not found: ground_truth*.json
   Make sure the files are in the working directory.


SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Compare LLM responses with expert classifications
result = compare_results(
    chat_responses,
    ground_truth,
    requirement_index=requirement_index,
)

metrics = calculate_metrics(result, field=CLASSIFICATION_FIELD)
report_path = f'{results_folder}/{REPORT_FILE_STANDARD}'
generate_report(
    result,
    metrics,
    report_path,
    input_metadata={
        'ground_truth_file': truth_file,
        'prompt_file': prompt_file,
        'input_file': input_file,
        'answer_files': chat_files,
        'combined_file': combined_chat_responses_path,
    }
)


# Wizualizacja


In [ ]:
# plot_confusion_matrix(metrics)


In [ ]:
# visualize_results(result)


# Send to MLflow


In [ ]:
import mlflow


In [ ]:
def log_artifact_as(mlflow, artifact_file_name, standard_file_name):
    """Log a local file to MLflow under a standard artifact name."""

    src = Path(artifact_file_name)
    if not src.exists():
        raise FileNotFoundError(f"File does not exist: {src}")

    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir) / standard_file_name
        tmp_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, tmp_path)

        artifact_path = str(Path(standard_file_name).parent)
        if artifact_path == ".":
            artifact_path = None

        mlflow.log_artifact(str(tmp_path), artifact_path=artifact_path)


def extract_version(variant_name: str, standard_name: str):
    """Return a filename suffix version relative to a standard artifact name."""
    variant_stem = Path(variant_name).stem
    standard_stem = Path(standard_name).stem
    suffix = variant_stem[len(standard_stem):]
    if suffix.startswith("_"):
        suffix = suffix[1:]
    return suffix if suffix else None


def dataframe_to_mlflow_dict(df):
    """Convert a DataFrame into an MLflow-serializable dictionary."""
    return {
        'index': [str(value) for value in df.index.tolist()],
        'columns': [str(value) for value in df.columns.tolist()],
        'data': df.fillna(0).astype(int).values.tolist(),
    }


def send_to_mlflow(
    result,
    metrics,
    chat_responses,
    ground_truth_file,
    answer_files,
    author,
    platform,
    model,
    dataset,
    project_space,
    documentation,
    prompt_file,
    input_file,
    results_folder,
    prompt_version,
    tracking_uri=None,
    experiment_name='ComparatorAI',
    run_name=None
):
    """
    Send comparison results to an MLflow server.

    Returns:
        run_id: MLflow run identifier
    """

    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    mlflow.set_experiment(experiment_name)

    if run_name is None:
        run_name = f"{platform}_{model}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    num_answers = len(chat_responses)
    num_standard = sum(1 for r in chat_responses if normalize_category(r.get(PREDICTED_CLASSIFICATION_FIELD)) == 'STANDARD')
    num_customization = sum(1 for r in chat_responses if normalize_category(r.get(PREDICTED_CLASSIFICATION_FIELD)) == 'CUSTOMIZATION')
    num_missing_answers = sum(1 for r in chat_responses if normalize_category(r.get(PREDICTED_CLASSIFICATION_FIELD)) == NO_ANSWER)

    prompt_content = ''
    if prompt_file and os.path.exists(prompt_file):
        with open(prompt_file, 'r', encoding='utf-8') as f:
            prompt_content = f.read()

    prompt_file_version = extract_version(prompt_file, PROMPT_FILE_STANDARD)
    ground_truth_file_version = extract_version(ground_truth_file, GROUND_TRUTH_STANDARD)
    input_file_version = extract_version(input_file, INPUT_FILE_STANDARD)

    with mlflow.start_run(run_name=run_name) as run:
        for key, val in {
            'author': author,
            'platform': platform,
            'model': model,
            'dataset': dataset,
            'documentation': documentation,
            'prompt_version': prompt_version,
            'prompt_file_version': prompt_file_version,
            'project_space': project_space,
            'input_version': input_file_version,
            'reference_file': ground_truth_file,
            'reference_file_version': ground_truth_file_version,
            'answer_files': ', '.join([os.path.basename(f) for f in answer_files]),
        }.items():
            mlflow.log_param(key, val)

        for key, val in {
            'num_answers': num_answers,
            'num_standard': num_standard,
            'num_customization': num_customization,
            'num_missing_answers': num_missing_answers,
        }.items():
            mlflow.log_metric(key, val)

        for key, val in {
            'all_Accuracy': metrics['global']['accuracy'],
            'all_Recall': metrics['overall_macro']['recall'],
            'all_Precision': metrics['overall_macro']['precision'],
            'all_F1': metrics['overall_macro']['f1-score'],
            'all_MCC': metrics['global']['mcc'],
            'all_Kappa': metrics['global']['kappa'],
        }.items():
            mlflow.log_metric(key, val)

        for threshold, m in metrics['per_confidence'].items():
            if m:
                for key, val in {
                    f'conf_{threshold}_Accuracy': m['accuracy'],
                    f'conf_{threshold}_MCC': m['mcc'],
                    f'conf_{threshold}_Kappa': m['kappa'],
                    f'conf_{threshold}_Precision_macro': m['macro_precision'],
                    f'conf_{threshold}_Recall_macro': m['macro_recall'],
                    f'conf_{threshold}_F1_macro': m['macro_f1'],
                    f'conf_{threshold}_Precision_wt': m['weighted_precision'],
                    f'conf_{threshold}_Recall_wt': m['weighted_recall'],
                    f'conf_{threshold}_F1_wt': m['weighted_f1'],
                    f'conf_{threshold}_N': m['num_samples'],
                }.items():
                    mlflow.log_metric(key, val)

        for category, cat_metrics in metrics['per_category'].items():
            cat_name = category.replace(' ', '_')
            mlflow.log_metric(f'all_{cat_name}_Recall', cat_metrics['recall'])
            mlflow.log_metric(f'all_{cat_name}_Precision', cat_metrics['precision'])
            mlflow.log_metric(f'all_{cat_name}_F1', cat_metrics['f1-score'])

        mlflow.log_text(prompt_content, PROMPT_FILE_STANDARD)
        mlflow.log_dict(dataframe_to_mlflow_dict(metrics['confusion_matrix']), 'confusion_matrix_all.json')
        log_artifact_as(mlflow, ground_truth_file, GROUND_TRUTH_STANDARD)
        log_artifact_as(mlflow, input_file, INPUT_FILE_STANDARD)
        log_artifact_as(
            mlflow,
            f'{results_folder}/{COMBINED_CHAT_RESPONSES_FILE_STANDARD}',
            COMBINED_CHAT_RESPONSES_FILE_STANDARD
        )
        log_artifact_as(mlflow, f'{results_folder}/{REPORT_FILE_STANDARD}', REPORT_FILE_STANDARD)
        mlflow.log_dict(result, 'comparison_results.json')

        print('✅ Results sent to MLflow!')
        print(f'   Run ID:       {run.info.run_id}')
        print(f'   Run Name:     {run_name}')
        print(f'   Experiment:   {experiment_name}')
        print(f'   Tracking URI: {mlflow.get_tracking_uri()}')

        return run.info.run_id


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                    MLFLOW CONFIGURATION                         ║
# ╚══════════════════════════════════════════════════════════════════╝

# Leave empty/None to use MLflow's default local tracking URI.
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI') or None
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'Krzysiek Experiments')

# Static parameters for MLflow
# PLATFORM options: 'GPT', 'Claude', 'Gemini', 'Perplexity', 'Kimi', 'Qwen', 'DeepSeek', 'MiniMax', 'Z'
PLATFORM = 'Claude'

# MODEL options: 'GPT 5.2', 'GPT 5.4', 'Opus 4.6', 'Opus 4.5', 'Opus 3', 'Sonnet 4.6', 'Sonnet 4.5',
#                'Haiku 4.5', 'Kimi 2.5 Thinking', 'DeepSeek-V3.2', 'Qwen3.5-Plus',
#                'Gemini 3 Fast', 'Gemini 3 Thinking', 'Gemini 3 Pro',
#                'GLM-5', 'GLM-5-Deep', 'Sonar', 'Nemotron 3 Super', 'MiniMax-M2.7-Air'
#                (append ' Thinking' for thinking variants)
MODEL = 'Opus 4.6 Thinking'

# Dataset split for stability studies
DATASET = "TEST"
# DATASET = "VALIDATION"

PROMPT_VERSION = '9'    # prompt version - use 'test' when working on a new prompt

PROJECT_SPACE = 'ERP XL Requirements - MD parts v9'
DOCUMENTATION = 'ERP XL Documentation - split 1-7'
AUTHOR = 'Krzysiek'


In [ ]:
run_id = send_to_mlflow(
    result=result,
    metrics=metrics,
    chat_responses=chat_responses,
    ground_truth_file=truth_file,
    answer_files=chat_files,
    author=AUTHOR,
    platform=PLATFORM,
    model=MODEL,
    dataset=DATASET,
    project_space=PROJECT_SPACE,
    documentation=DOCUMENTATION,
    prompt_file=prompt_file,
    input_file=input_file,
    results_folder=results_folder,
    prompt_version=PROMPT_VERSION,
    tracking_uri=MLFLOW_TRACKING_URI,
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    run_name=None,
)
